# Logistic regression: loan default classifier

Part of the ML Portfolio Model Zoo. Educational purposes only.

Given a borrower's income, debt-to-income ratio, credit history, recent late payments and employment type, estimate the probability of default. The data is **synthetic**, generated from a known default process, so nothing sensitive is used and the result can be sanity-checked.

In [1]:
import pandas as pd
from train import *

# Everything here reuses train.py, so the notebook and the site always use the same pipeline.

## Data

In [2]:
data = make_data()
print(data.shape)
data.head()

(6000, 6)


,annual_income_k,debt_to_income_pct,credit_history_years,late_payments_12m,employment,default
0,70.0,21.0,1.0,1,self-employed,0
1,36.0,14.0,7.0,2,salaried,0
2,87.0,5.0,3.0,1,salaried,0
3,96.0,10.0,13.0,0,salaried,0
4,23.0,39.0,9.0,1,self-employed,0


In [3]:
data.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
annual_income_k,6000.0,NaN,NaN,NaN,67.7395,36.195142,10.0,43.0,60.0,83.0,300.0
debt_to_income_pct,6000.0,NaN,NaN,NaN,26.198667,13.556296,1.0,16.0,25.0,35.0,75.0
credit_history_years,6000.0,NaN,NaN,NaN,8.780833,5.649494,0.0,5.0,8.0,12.0,40.0
late_payments_12m,6000.0,NaN,NaN,NaN,0.793167,0.893226,0.0,0.0,1.0,1.0,5.0
employment,6000,3,salaried,3873,NaN,NaN,NaN,NaN,NaN,NaN,NaN
default,6000.0,NaN,NaN,NaN,0.1655,0.371662,0.0,0.0,0.0,0.0,1.0


## Train and evaluate

`train_and_evaluate()` fits the scikit-learn pipeline, scores it on a held-out split, exports it to ONNX and asserts that onnxruntime reproduces scikit-learn's predictions.

In [4]:
result = train_and_evaluate()
print(f"accuracy {result.metrics['accuracy']:.3f}   ROC AUC {result.metrics['roc_auc']:.3f}   Brier {result.metrics['brier']:.3f}")
print(f"base default rate {result.metrics['default_rate']:.1%}")
print(f"ONNX vs sklearn max abs diff: {result.metrics['onnx_max_abs_diff']:.2e}")

accuracy 0.855   ROC AUC 0.814   Brier 0.106
base default rate 16.6%
ONNX vs sklearn max abs diff: 8.78e-08


## Predict with the exported ONNX model

In [5]:
example = pd.DataFrame([PRESETS[1]["values"]])[ALL_COLUMNS]
labels, probs = onnx_predict(result.model_bytes, example)
print("High-risk preset ->", LABELS[int(labels[0])], f"(P(default) = {probs[0][1]:.1%})")

High-risk preset -> Default (P(default) = 97.9%)


## Site artefacts

Run `uv run python train.py` to write `model.onnx`, `schema.json`, the charts and the .NET parity fixtures into the web app. See the README for details.